In [80]:
import pandas as pd
from thefuzz import process
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# For imputation
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

In [81]:
train_df = pd.read_csv("data/train.csv")

In [82]:
# String cleaning and Small numbers changes

def simple_processing(df):
    """
    Apply string cleaning, brand/model corrections, and fuzzy matching.
    These operations don"t require fitting on training data.
    """

    df = df.copy()
    # ============================================================================
    # SECTION 1: REFERENCE DATA SETUP
    # ============================================================================
    
    # Reference list of correct model names
    models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", 
              "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", 
              "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", 
              "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", 
              "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", 
              "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", 
              "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", 
              "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", 
              "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", 
              "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", 
              "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", 
              "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", 
              "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", 
              "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", 
              "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", 
              "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", 
              "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", 
              "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", 
              "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", 
              "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]
    
    # Get unique short model names (2 characters) for separate handling
    short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
    short_models = list(set(short_models))
    
    transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
    fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]
    
    # Brand name corrections mapping
    brand_mapping = {
        "vw": "vw",
        "v": "vw",
        "w": "vw",
        
        "toyota": "toyota",
        "toyot": "toyota",
        "oyota": "toyota",
        
        "audi": "audi",
        "aud": "audi",
        "udi": "audi",
        "ud": "audi",
        
        "ford": "ford",
        "for": "ford",
        "ord": "ford",
        "or": "ford",
        
        "bmw": "bmw",
        "bm": "bmw",
        "mw": "bmw",
        
        "skoda": "skoda",
        "skod": "skoda",
        "koda": "skoda",
        "kod": "skoda",
        
        "opel": "opel",
        "ope": "opel",
        "pel": "opel",
        "pe": "opel",
        
        "mercedes": "mercedes",
        "mercede": "mercedes",
        "ercedes": "mercedes",
        "ercede": "mercedes",
        
        "hyundai": "hyundai",
        "hyunda": "hyundai",
        "yundai": "hyundai",
        "yunda": "hyundai"
    }
    
    # ============================================================================
    # SECTION 2: INITIAL CLEANING (NO FITTING REQUIRED) -> no risk of data leakage
    # ============================================================================
        
    # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column
    df["Brand"] = df["Brand"].str.lower().str.strip()
    df["model"] = df["model"].str.lower().str.strip()
    df["transmission"] = df["transmission"].str.lower().str.strip()
    df["fuelType"] = df["fuelType"].str.lower().str.strip()
    
    # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)
    df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN")
    
    # 1.1 Fixing brands
    df["Brand"] = df["Brand"].map(brand_mapping)
    
    # 1.2 Fixing models
    # Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)
    # Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz
    
    # VECTORIZED APPROACH: Only perform fuzzy matching once per unique value instead of per row
    
    # Models - handle different lengths separately
    unique_models = df["model"].unique()
    model_lookup = {}
    for val in unique_models:
        if pd.isna(val) or val == "NaN":
            model_lookup[val] = "NaN"
        elif len(val) > 2:  # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
            model_lookup[val] = process.extractOne(val, models)[0]  # [0] because we get the name and score as a return -> score used for debugging
        elif len(val) == 2:  # Use the short names list for comparisons if the model names are 2 letters
            model_lookup[val] = process.extractOne(val, short_models)[0]
        else:  # We can define models with only one letter
            model_lookup[val] = "NaN"
    df["model"] = df["model"].map(model_lookup)
    
    # Transmission
    unique_trans = df["transmission"].unique()
    trans_lookup = {val: process.extractOne(val, transmission_types)[0] for val in unique_trans}
    df["transmission"] = df["transmission"].map(trans_lookup)
    
    # FuelType
    unique_fuel = df["fuelType"].unique()
    fuel_lookup = {val: process.extractOne(val, fuel_types)[0] for val in unique_fuel}
    df["fuelType"] = df["fuelType"].map(fuel_lookup)
    
    # Convert the str NaN values back to pd.NA for easier further processing and readability
    df["model"] = df["model"].replace("NaN", pd.NA)
    df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
    df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)
    
    # Get the most frequent brand for each model -> returns df with model and brand
    brand_models = df.groupby("model")["Brand"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else pd.NA)
    df = pd.merge(df, brand_models, on="model", how="left", suffixes=("", "_mode"))  # add the model and brand df to our main df (onyl add the brand columns, join on model)
    
    df["Brand"] = df["Brand"].fillna(df["Brand_mode"])  # rename new column
    df.drop("Brand_mode", axis=1, inplace=True)  # remove the old brand column
    
    ################################################################################
    # Simple Number Cleaning
    ################################################################################

    # Cleaning numeric columns
    df["year"] = df["year"].round(0)
    
    # Create the new column
    df["stated_no_damage"] = ~df["hasDamage"].astype(bool)
    df = df.drop(["hasDamage"], axis=1)

    # Round year to integer (no fractional years)
    df["year"] = df["year"].round()
    
    # Mileage: take absolute value and round
    # Some imputation might produce small negative values
    df["mileage"] = abs(df["mileage"].round())
    
    # Tax: take absolute value and round
    df["tax"] = abs(df["tax"].round())
    
    # MPG: round to 1 decimal place 
    df["mpg"] = abs(df["mpg"].round(1))
    
    # Engine size: round to 1 decimal place
    df["engineSize"] = abs(df["engineSize"].round(1))
    
    # Paint quality correction (domain-specific business logic)
    # Assumption based on data exploration:
    # - Values < 4 likely had decimal point in wrong place (e.g., 3.5 -> 35%)
    # - Values > 100 likely have erroneous leading 1 (e.g., 185 -> 85%)
    def fix_paint_quality(x):
        if x < 4:
            return x * 10
        elif x > 100:
            return x - 100
        else:
            return x
    
    df["paintQuality%"] = df["paintQuality%"].apply(fix_paint_quality).round()
    
    # Previous owners: take absolute value and round to integer
    df["previousOwners"] = abs(df["previousOwners"].round())
    
    return df

In [83]:
# Categorical featue Encoding
def fit_transform_encoding(df):
    """
    Fit label encoders for categorical columns on training data.
    """
    
    encoders = {
        "Brand": LabelEncoder(),
        "model": LabelEncoder(),
        "transmission": LabelEncoder(),
        "fuelType": LabelEncoder()
    }
    
    # Fit each encoder on the corresponding column
    """encoders["brand"].fit(df["Brand"])
    encoders["model"].fit(df["model"])
    encoders["transmission"].fit(df["transmission"])
    encoders["fuelType"].fit(df["fuelType"])"""

    columns = ["Brand", "model", "transmission", "fuelType"]

    # Code adapted from: https://stackoverflow.com/questions/36808434/label-encoder-encoding-missing-values

    for column in columns:
        # Get non-null string values
        mask = df[column].notna() & (df[column].apply(type) == str)
        
        # Fit encoder on unique non-null values
        fit_by = df.loc[mask, column].unique()
        encoders[column].fit(fit_by)
        
        # Transform only non-null values (vectorized)
        new_col_name = column + "_transformed"
        df[new_col_name] = pd.NA  # Initialize with NA
        df.loc[mask, new_col_name] = encoders[column].transform(df.loc[mask, column])
        
        # Convert to nullable integer
        df[new_col_name] = df[new_col_name].astype("Int64")

    df = df.drop(columns, axis=1)
    return df, encoders

In [84]:
# Train imputer on train

def fit_imputer(df, fast=True): 
    # Select estimator based on speed/accuracy tradeoff
    if fast:
        # Use default BayesianRidge (fast, ~1 second)
        estimator = None
    else:
        # Use Random Forest for better accuracy with complex relationships (~2 minutes)
        estimator = RandomForestRegressor(
            n_estimators=20,      # Limited trees for speed
            max_depth=10,         # Prevent overfitting
            random_state=12       # Reproducibility
        )

    # TODO: for later
    # Test if we perform better if we use numerical and categorical imputers separately
    
    # Initialize imputer
    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="mean"         # Initial fill before iterative process
    )
    
    # TODO: Test other categorical imputers like: missForest, datawig
    """imputer = IterativeImputer(
        estimator=RandomForestClassifier(),
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="most_frequent"         # Initial fill before iterative process
        )"""

    # FIT on training data
    # CRITICAL: We fit on data that still has missing values!
    # The imputer learns patterns of missingness and relationships

    imputer.fit(df)
    
    return imputer

In [85]:
def apply_imputer(df, imputer):
    """
        Apply the pretrained imputer to the dataframe
    """

    imputed_values = imputer.transform(df)

    df[df.columns] = imputed_values

    # TODO: rounding is not the best approach as the imputers prediction are continues thus 1.2 doesnt mean the value is closer to 1 than 2
    # However, rounding is the quickest way to fix this for now
    df[["mpg", "engineSize"]] = df[["mpg", "engineSize"]].round(1)
    try: 
        df[["year", "price", "tax", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = df[["year", "price", "tax", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]].round(0).astype(int)
    except KeyError:
        pass
    return df

In [86]:
def decode(df, encoders):
    # Iterative imputer produces ~20-30 values that are outside of the range of the encoder
    # The simplest fix is to clip does values back into the range of the encoder
 

    df["Brand_transformed"] = df["Brand_transformed"].clip(lower=0, upper=encoders["Brand"].classes_.shape[0]-1).astype(int)
    df["Brand"] = encoders["Brand"].inverse_transform(df["Brand_transformed"])

    df["transmission_transformed"] = df["transmission_transformed"].clip(lower=0, upper=encoders["transmission"].classes_.shape[0]-1).astype(int)
    df["transmission"] = encoders["transmission"].inverse_transform(df["transmission_transformed"])
        
    # Use clip with the information of the fitted encoder, .classes_.shape gives us the dimension of the labels the encoder uses [0] is the rows - 1 because we start clipping at 0
    df["model_transformed"] = df["model_transformed"].clip(lower=0, upper=encoders["model"].classes_.shape[0]-1).astype(int)
    df["model"] = encoders["model"].inverse_transform(df["model_transformed"])
    
    df["fuelType_transformed"] = df["fuelType_transformed"].clip(lower=0, upper=encoders["fuelType"].classes_.shape[0]-1).astype(int)
    df["fuelType"] = encoders["fuelType"].inverse_transform(df["fuelType_transformed"])
    

    #df.drop(columns=["Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"], inplace=True)

    return df

# Workflow for Train and Validation Sets

In [87]:
# Fix typos and small numeric cleanup
df = simple_processing(train_df)
# Encode cateogrical columns and replace the str with int columns. Return fitted encoders for decoding at the end
df_encoded, encoders = fit_transform_encoding(df)

# Train validation split
train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42)

# Train imputer on train (data leakage risk thus only train on train_split)
imputer = fit_imputer(train_split)

# Apply trained imputer to both datasplits
imputed_train = apply_imputer(train_split, imputer)
imputer_test = apply_imputer(validation_split, imputer)

# Decode encoded columns using the fitted encoders
train_processed = decode(imputed_train, encoders)
validation_processed = decode(imputer_test, encoders)

# Workflow for Seperated Testing Dataset

In [88]:
test_df = pd.read_csv("data/test.csv")

In [ ]:
test = simple_processing(test_df)
test_df_encoded, test_encoders = fit_transform_encoding(test)


df_encoded_no_price = df_encoded.drop(["price"], axis=1)
test_imputers = fit_imputer(df_encoded_no_price) # we use the full encoded training dataframe to train the imputers

# Apply trained imputer to both datasplits
test_df_imputed = apply_imputer(test_df_encoded, test_imputers)

test_processed = decode(test_df_imputed, test_encoders)
